# Predictive Pricing Analytics using Machine Learning

## Feature Engineering

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

In [4]:
with zipfile.ZipFile(r"C:\Users\ANTARA\Downloads\archive (2).zip") as z:
    train_df = pd.read_csv(z.open("train.csv"))
    store_df = pd.read_csv(z.open("store.csv"))

C:\Users\ANTARA\AppData\Local\Temp\ipykernel_17972\1137355779.py:2: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(z.open("train.csv"))


# Data Preparation

In [5]:
train_df["Date"] = pd.to_datetime(train_df["Date"])

store_df["CompetitionDistance"] = store_df["CompetitionDistance"].fillna(
    store_df["CompetitionDistance"].median()
)

store_df["CompetitionOpenSinceMonth"] = store_df["CompetitionOpenSinceMonth"].fillna(0)
store_df["CompetitionOpenSinceYear"] = store_df["CompetitionOpenSinceYear"].fillna(0)
store_df["Promo2SinceWeek"] = store_df["Promo2SinceWeek"].fillna(0)
store_df["Promo2SinceYear"] = store_df["Promo2SinceYear"].fillna(0)
store_df["PromoInterval"] = store_df["PromoInterval"].fillna("No Promo")

# Merge the Datasets

In [6]:
pricing_df = pd.merge(train_df, store_df, on="Store", how="left")

In [7]:
pricing_df["Year"] = pricing_df["Date"].dt.year

pricing_df[["Date", "Year"]].head()

,Date,Year
0,2015-07-31,2015
1,2015-07-31,2015
2,2015-07-31,2015
3,2015-07-31,2015
4,2015-07-31,2015


In [8]:
pricing_df["Date"].dt.year

0          2015
1          2015
2          2015
3          2015
4          2015
           ... 
1017204    2013
1017205    2013
1017206    2013
1017207    2013
1017208    2013
Name: Date, Length: 1017209, dtype: int32

# Create Month Feature

The Month feature is extracted from the Date column to capture seasonal sales patterns and monthly trends.

In [11]:
pricing_df["Month"] = pricing_df["Date"].dt.month

In [12]:
pricing_df[["Date", "Month"]].head()

,Date,Month
0,2015-07-31,7
1,2015-07-31,7
2,2015-07-31,7
3,2015-07-31,7
4,2015-07-31,7


# Create Day Feature

The Day feature is extracted from the Date column to capture daily sales patterns and identify variations in customer purchasing behavior throughout the month.

In [13]:
pricing_df["Day"] = pricing_df["Date"].dt.day

In [14]:
pricing_df[["Date", "Day"]].head()

,Date,Day
0,2015-07-31,31
1,2015-07-31,31
2,2015-07-31,31
3,2015-07-31,31
4,2015-07-31,31


# Create Day of Week Feature

The DayOfWeek feature is extracted to identify differences in sales across weekdays and weekends.

In [15]:
pricing_df["DayOfWeek"] = pricing_df["Date"].dt.dayofweek

In [16]:
pricing_df[["Date", "DayOfWeek"]].head()

,Date,DayOfWeek
0,2015-07-31,4
1,2015-07-31,4
2,2015-07-31,4
3,2015-07-31,4
4,2015-07-31,4


# Create Week of Year Feature

The WeekOfYear feature captures weekly sales trends and seasonal demand throughout the year.

In [17]:
pricing_df["WeekOfYear"] = pricing_df["Date"].dt.isocalendar().week.astype(int)

In [18]:
pricing_df[["Date", "WeekOfYear"]].head()

,Date,WeekOfYear
0,2015-07-31,31
1,2015-07-31,31
2,2015-07-31,31
3,2015-07-31,31
4,2015-07-31,31


# Create Quarter Feature

The Quarter feature is extracted from the Date column to capture quarterly business trends and seasonal variations in sales.

In [19]:
pricing_df["Quarter"] = pricing_df["Date"].dt.quarter

In [20]:
pricing_df[["Date", "Quarter"]].head()

,Date,Quarter
0,2015-07-31,3
1,2015-07-31,3
2,2015-07-31,3
3,2015-07-31,3
4,2015-07-31,3


# Create Weekend Feature

The Weekend feature identifies whether a particular date falls on a weekend. This helps capture differences in customer shopping behavior between weekdays and weekends.

In [21]:
pricing_df["Weekend"] = pricing_df["DayOfWeek"].apply(lambda x: 1 if x >= 5 else 0)

In [22]:
pricing_df[["Date", "DayOfWeek", "Weekend"]].head()

,Date,DayOfWeek,Weekend
0,2015-07-31,4,0
1,2015-07-31,4,0
2,2015-07-31,4,0
3,2015-07-31,4,0
4,2015-07-31,4,0


# Create Competition Age Feature

The CompetitionAge feature represents the number of years since the nearest competitor opened. It helps evaluate how long a store has faced competition.

In [23]:
pricing_df["CompetitionAge"] = (
    pricing_df["Year"] - pricing_df["CompetitionOpenSinceYear"]
)

In [24]:
pricing_df["CompetitionAge"] = pricing_df["CompetitionAge"].clip(lower=0)

In [25]:
pricing_df[
    ["CompetitionOpenSinceYear", "Year", "CompetitionAge"]
].head()

,CompetitionOpenSinceYear,Year,CompetitionAge
0,2008.0,2015,7.0
1,2007.0,2015,8.0
2,2006.0,2015,9.0
3,2009.0,2015,6.0
4,2015.0,2015,0.0


# Create Promo Duration Feature

The PromoDuration feature estimates how long a promotional campaign has been active. This feature helps understand the long-term impact of promotions on sales.

In [26]:
pricing_df["PromoDuration"] = (
    pricing_df["Year"] - pricing_df["Promo2SinceYear"]
)

In [27]:
pricing_df["PromoDuration"] = pricing_df["PromoDuration"].clip(lower=0)

In [28]:
pricing_df[
    ["Promo2SinceYear", "Year", "PromoDuration"]
].head()

,Promo2SinceYear,Year,PromoDuration
0,0.0,2015,2015.0
1,2010.0,2015,5.0
2,2011.0,2015,4.0
3,0.0,2015,2015.0
4,0.0,2015,2015.0


# Label Encoding

Label Encoding converts categorical (text) variables into numerical values so that machine learning algorithms can process them effectively.

In [30]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.3 MB 4.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.3 MB 7.1 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.3 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 10.1 MB/s  0:00:00
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.8/37.3 MB 9.1 MB/s eta 0:00:04
   --- ------------------------------------ 3.7/37.3 MB 8.7 MB/s eta 0:00:04
   ----- ---------------------------------- 5.0/37.3 MB 7.7 MB/s eta 0:00:05
   ------ --------------------------------- 5.8/37.3 MB 7.1 MB/s eta 0:00:05
   -------- ------------------------------- 7.6/37.3 MB 7.2 MB/s eta 0:00:05
   --------- ------------------------------ 8.7/37.3 MB 6.9 MB/s eta 0:00:05
   ---------- ------

In [1]:
from sklearn.preprocessing import LabelEncoder

In [6]:
import pandas as pd
import zipfile

In [7]:
with zipfile.ZipFile(r"C:\Users\ANTARA\Downloads\archive (2).zip") as z:
    train_df = pd.read_csv(z.open("train.csv"))
    store_df = pd.read_csv(z.open("store.csv"))

C:\Users\ANTARA\AppData\Local\Temp\ipykernel_16296\1137355779.py:2: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(z.open("train.csv"))


In [8]:
train_df["Date"] = pd.to_datetime(train_df["Date"])

In [9]:
store_df["CompetitionDistance"] = store_df["CompetitionDistance"].fillna(
    store_df["CompetitionDistance"].median()
)

store_df["CompetitionOpenSinceMonth"] = store_df["CompetitionOpenSinceMonth"].fillna(0)
store_df["CompetitionOpenSinceYear"] = store_df["CompetitionOpenSinceYear"].fillna(0)
store_df["Promo2SinceWeek"] = store_df["Promo2SinceWeek"].fillna(0)
store_df["Promo2SinceYear"] = store_df["Promo2SinceYear"].fillna(0)
store_df["PromoInterval"] = store_df["PromoInterval"].fillna("No Promo")

In [10]:
pricing_df = pd.merge(train_df, store_df, on="Store", how="left")

In [11]:
pricing_df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,No Promo
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,No Promo


# Label Encoding

Convert categorical features into numerical values so they can be used by machine learning algorithms.

In [12]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "StoreType",
    "Assortment",
    "StateHoliday",
    "PromoInterval"
]

for column in categorical_columns:
    le = LabelEncoder()
    pricing_df[column] = le.fit_transform(pricing_df[column].astype(str))

In [13]:
pricing_df.to_csv("pricing_data_processed.csv", index=False)

In [14]:
pricing_df = pd.read_csv("pricing_data_processed.csv")